# 第11章 注目する仕組み：Self-Attention

どの文字に注目するかを、文字どうしが話し合って決める

教科書のページ：https://sakunokyumei.github.io/llm-research-textbook/11-attention.html

上のセルから順に、再生ボタン（▶）か `Shift + Enter` で実行します。説明は教科書のページにあります。


## 準備と、新しい問題の出し方


In [ ]:
import re, math, urllib.request
import torch
import torch.nn as nn
import torch.nn.functional as F

def load_soseki(file_id="773_ruby_5968"):
    url = ("https://raw.githubusercontent.com/aozorahack/aozorabunko_text/master/"
           f"cards/000148/files/{file_id}/{file_id}.txt")
    text = urllib.request.urlopen(url).read().decode("cp932")
    text = text.split("-------------------------------------------------------")[2]
    text = text.split("底本：")[0]
    text = re.sub(r"《.*?》", "", text)
    text = re.sub(r"※?［＃.*?］", "", text)
    text = text.replace("｜", "").replace("\r\n", "\n")
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()

text = load_soseki()
chars = sorted(set(text))
V = len(chars)
stoi = {ch: i for i, ch in enumerate(chars)}
itos = {i: ch for ch, i in stoi.items()}
data = torch.tensor([stoi[c] for c in text])
n = int(len(data) * 0.9)
train_data, val_data = data[:n], data[n:]

def get_batch(data, batch_size, T, g):
    ix = torch.randint(0, len(data) - T, (batch_size,), generator=g)
    x = torch.stack([data[i:i + T] for i in ix])
    y = torch.stack([data[i + 1:i + T + 1] for i in ix])   # 1文字ずらしたもの
    return x, y

g = torch.Generator().manual_seed(0)
x, y = get_batch(train_data, 1, 8, g)
print("入力：", repr("".join(itos[i] for i in x[0].tolist())))
print("正解：", repr("".join(itos[i] for i in y[0].tolist())))

## ステップ1：過去を「平均」してみる


In [ ]:
torch.manual_seed(0)
B, T, C = 4, 8, 2            # 4個の文章、8文字、各文字は2個の数
x = torch.randn(B, T, C)

xbow = torch.zeros(B, T, C)  # bow：bag of words（言葉の袋）
for b in range(B):
    for t in range(T):
        xbow[b, t] = x[b, :t + 1].mean(dim=0)   # 位置0〜tの平均

print(x[0])
print(xbow[0])

## ステップ2：行列の掛け算で一気に


In [ ]:
wei = torch.tril(torch.ones(T, T))        # 下三角行列（左下だけ1）
wei = wei / wei.sum(dim=1, keepdim=True)  # 各行の合計が1になるように
print(wei)
xbow2 = wei @ x                           # [T, T] @ [B, T, C] → [B, T, C]
print(torch.allclose(xbow, xbow2))

## ステップ3：softmaxで重みを作る


In [ ]:
tril = torch.tril(torch.ones(T, T))
scores = torch.zeros(T, T)                                # 今はまだ全部0点
scores = scores.masked_fill(tril == 0, float("-inf"))     # 未来の位置は -∞ 点
wei = F.softmax(scores, dim=-1)
print(wei[:3])
print(torch.allclose(wei @ x, xbow))

## 内積：似ている度


In [ ]:
a = torch.tensor([1.0, 0.0])
print("同じ向き：", a @ torch.tensor([2.0, 0.0]))
print("直角：    ", a @ torch.tensor([0.0, 2.0]))
print("逆向き：  ", a @ torch.tensor([-2.0, 0.0]))

## ステップ4：Query・Key・Value


In [ ]:
torch.manual_seed(0)
B, T, C = 4, 8, 32
x = torch.randn(B, T, C)

head_size = 16
key = nn.Linear(C, head_size, bias=False)
query = nn.Linear(C, head_size, bias=False)
value = nn.Linear(C, head_size, bias=False)

k = key(x)      # [B, T, 16]
q = query(x)    # [B, T, 16]
scores = q @ k.transpose(-2, -1)    # [B, T, 16] @ [B, 16, T] → [B, T, T]
scores = scores.masked_fill(tril == 0, float("-inf"))
wei = F.softmax(scores, dim=-1)
out = wei @ value(x)                # [B, T, T] @ [B, T, 16] → [B, T, 16]

print(scores.shape, wei.shape, out.shape)
print(wei[0].detach().round(decimals=2))

## なぜ √d で割るのか


In [ ]:
torch.manual_seed(0)
for d in [16, 64, 256]:
    q = torch.randn(1000, d)
    k = torch.randn(1000, d)
    dots = (q * k).sum(dim=1)        # 1000組の内積
    print(f"d={d:3d}：内積の分散 {dots.var():7.2f}   √dで割ると {(dots / math.sqrt(d)).var():.2f}")

## 部品にまとめる：Head クラス


In [ ]:
class Head(nn.Module):
    def __init__(self, n_embd, head_size, block_size):
        super().__init__()
        self.key = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)
        self.register_buffer("tril", torch.tril(torch.ones(block_size, block_size)))

    def forward(self, x):
        B, T, C = x.shape
        k, q, v = self.key(x), self.query(x), self.value(x)
        scores = q @ k.transpose(-2, -1) / math.sqrt(k.shape[-1])
        scores = scores.masked_fill(self.tril[:T, :T] == 0, float("-inf"))
        self.wei = F.softmax(scores, dim=-1)       # 後で中身を見られるように取っておく
        return self.wei @ v

In [ ]:
torch.manual_seed(0)
head = Head(n_embd=32, head_size=16, block_size=8)
x1 = torch.randn(1, 8, 32)
x2 = x1.clone()
x2[0, 7] = torch.randn(32)            # 位置7（最後）だけ書き換える

out1, out2 = head(x1), head(x2)
print("位置0〜6の出力は同じ？", torch.allclose(out1[0, :7], out2[0, :7]))
print("位置7の出力は同じ？  ", torch.allclose(out1[0, 7], out2[0, 7]))

## Attentionだけで言語モデルを作る


In [ ]:
class OneHeadLM(nn.Module):
    def __init__(self, V, block_size, n_embd=64):
        super().__init__()
        self.tok = nn.Embedding(V, n_embd)           # 文字の埋め込み
        self.pos = nn.Embedding(block_size, n_embd)  # 位置の埋め込み（後で説明）
        self.head = Head(n_embd, n_embd, block_size)
        self.out = nn.Linear(n_embd, V)

    def forward(self, idx, targets=None):
        B, T = idx.shape
        x = self.tok(idx) + self.pos(torch.arange(T))
        x = x + self.head(x)              # 過去の情報を「足し込む」
        logits = self.out(x)
        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.view(B * T, -1), targets.view(B * T))
        return logits, loss

@torch.no_grad()
def val_loss(model, T):
    model.eval()
    xs = torch.stack([val_data[i:i + T] for i in range(0, len(val_data) - T - 1, T)])
    ys = torch.stack([val_data[i + 1:i + T + 1] for i in range(0, len(val_data) - T - 1, T)])
    loss = model(xs, ys)[1].item()
    model.train()
    return loss

block_size = 64
torch.manual_seed(0)
g = torch.Generator().manual_seed(0)
model = OneHeadLM(V, block_size)
opt = torch.optim.AdamW(model.parameters(), lr=3e-3)
print("パラメータの数：", sum(p.numel() for p in model.parameters()))

for step in range(2001):
    x, y = get_batch(train_data, 32, block_size, g)
    _, loss = model(x, y)
    opt.zero_grad()
    loss.backward()
    opt.step()
    if step % 500 == 0:
        print(f"ステップ {step:4d}：訓練 {loss.item():.3f}  検証 {val_loss(model, block_size):.3f}")

## 注目の様子をのぞく


In [ ]:
!pip install -q matplotlib-fontja

In [ ]:
import matplotlib.pyplot as plt
import matplotlib_fontja

sample = "私はその人を常に先生と呼んでいた。"
idx = torch.tensor([[stoi[c] for c in sample]])
model.eval()
with torch.no_grad():
    model(idx)
wei = model.head.wei[0]

plt.figure(figsize=(6, 5))
plt.imshow(wei, cmap="Oranges")
plt.xticks(range(len(sample)), list(sample))
plt.yticks(range(len(sample)), list(sample))
plt.xlabel("注目される文字（Key）")
plt.ylabel("注目する文字（Query）")
plt.title("学習したAttentionの重み")
plt.show()